<a href="https://colab.research.google.com/github/Nishan-vxm/Credit-card-froud-direction-/blob/main/Credit_card_froud_direction_UI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio scikit-learn joblib tensorflow

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving credit_card_fraud_ann.keras to credit_card_fraud_ann.keras
Saving credit_card_scaler.pkl to credit_card_scaler.pkl


In [ ]:
import os
import joblib
import tensorflow as tf

from tensorflow.keras.models import load_model

model_path = "/content/credit_card_fraud_ann.keras"
scaler_path = "/content/credit_card_scaler.pkl"

print("Model file exists:", os.path.exists(model_path))
print("Scaler file exists:", os.path.exists(scaler_path))

if os.path.exists(model_path) and os.path.exists(scaler_path):
    model = load_model(model_path)
    scaler = joblib.load(scaler_path)

    print("\n✅ Model loaded successfully!")
    print("✅ Scaler loaded successfully!")
    print("Model input shape:", model.input_shape)
else:
    print("\n❌ Files missing. Upload both files again.")

Model file exists: True
Scaler file exists: True

✅ Model loaded successfully!
✅ Scaler loaded successfully!
Model input shape: (None, 30)


In [ ]:
import gradio as gr
import numpy as np
import pandas as pd

# Feature order must match the order used during training
feature_names = [
    "Time",
    "V1", "V2", "V3", "V4", "V5", "V6", "V7",
    "V8", "V9", "V10", "V11", "V12", "V13", "V14",
    "V15", "V16", "V17", "V18", "V19", "V20", "V21",
    "V22", "V23", "V24", "V25", "V26", "V27", "V28",
    "Amount"
]

def predict_fraud(*values):
    try:
        transaction = np.array(values, dtype=float).reshape(1, -1)

        # Apply the same scaler used during model training
        transaction_scaled = scaler.transform(transaction)

        prediction = model.predict(transaction_scaled, verbose=0)
        fraud_probability = float(np.asarray(prediction).ravel()[0])

        # This assumes the ANN outputs a sigmoid probability for Class=1
        if fraud_probability >= 0.5:
            status = "🚨 Potential Fraud Detected"
        else:
            status = "✅ Likely Legitimate Transaction"

        return (
            status,
            f"{fraud_probability * 100:.2f}%",
            "This is a model prediction, not a confirmed fraud determination."
        )

    except Exception as e:
        return "Prediction Error", "N/A", str(e)


with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        secondary_hue="blue",
        neutral_hue="slate"
    ),
    title="FraudGuard AI | Transaction Security"
) as demo:

    gr.Markdown("""
    # 🛡️ FraudGuard AI
    ### Credit Card Transaction Intelligence
    Analyze transaction features and flag potentially fraudulent activity.
    """)

    gr.Markdown(
        "**Model:** Artificial Neural Network (ANN) · "
        "**Input:** 30 features · **Threshold:** 0.50"
    )

    with gr.Accordion("Enter Transaction Features", open=True):
        inputs = []

        for name in feature_names:
            default_value = 0.0
            inputs.append(
                gr.Number(
                    label=name,
                    value=default_value,
                    precision=6
                )
            )

        gr.Markdown(
            "Note: V1–V28 are anonymized features. "
            "Use real preprocessed transaction rows for meaningful predictions."
        )

    predict_button = gr.Button(
        "🔍 Analyze Transaction",
        variant="primary"
    )

    with gr.Group():
        status_output = gr.Textbox(label="Prediction")
        probability_output = gr.Textbox(label="Predicted Fraud Probability")
        note_output = gr.Textbox(label="Important Note")

    predict_button.click(
        fn=predict_fraud,
        inputs=inputs,
        outputs=[status_output, probability_output, note_output]
    )

demo.launch(share=True)

/tmp/ipykernel_1789/3701599025.py:41: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb4f4c7a6262d162c6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
